# Decision Trees — Interpretable Models with Sharp Edges

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb11_decision_trees.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how a decision tree partitions feature space using axis-aligned splits, for **both classification and regression** targets.
2. Fit and visualize a `DecisionTreeClassifier` (Wisconsin Breast Cancer) and a `DecisionTreeRegressor` (California Housing) using `plot_tree`.
3. Diagnose **overfitting** in a tree by comparing training accuracy / R² against 5-fold cross-validation, and read the train-vs-CV gap as a regularization signal.
4. Compare a tuned tree against its linear analogue (Logistic Regression vs Ridge) on identical CV folds.
5. Choose the simplest competitive `max_depth` using the **one-standard-error rule** — the discipline you will reuse in nb12, nb13, and nb14's selection ceremony.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track and Exercise 2 on the regression track. Complete both before submitting your notebook.

---

## 💼 Why This Matters

This notebook runs **two parallel cases** end-to-end, because your final-project group's target may be either classification or regression. Watching the same algorithm work on both cuts your transfer cost in half.

### The classification case — State Health Department screening

Wisconsin Breast Cancer biopsy data, 569 patients, 30 cell-nucleus measurements per biopsy, target = malignant vs benign. The Department's existing screening tool flags about 10% of clinic visits for confirmatory testing; the screening logistic-regression model from nb06 lifts true-positive rate but the oncologists push back: *"I can't explain to a patient why the model flagged them. I need to see the decision logic."*

A decision tree gives them: **"if worst radius > 16.8 AND mean texture > 21.4, then malignant."** The path from root to leaf is an auditable flowchart any clinician can read aloud.

### The regression case — HomeValue Analytics property listings

California Housing data, 20,640 census tracts, 8 features (median income, house age, occupancy, location), target = median house value in USD 100K units. HomeValue Analytics wants to deploy a price-prediction model on its public-facing listings: *"What is this property worth, given comparable tracts in the area?"*

A regression tree gives the legal team something they can defend in court: **"this property's prediction is the average of 47 comparable tracts that match it on the seven splits below."** No coefficients to interpret, no extrapolation outside the training range — just an averaging rule with a documented provenance.

### The reference floor every tree-based model must clear

Week 2 closed with a disciplined comparison: tuned linear models versus their default counterparts under nb09's **CI-overlap test**. The verdict was unambiguous on both spines:

- **Classification:** `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])` — random-search over `C` produced no statistically clear winner (the tuned variant's CI overlaps the default's). The default ships.
- **Regression:** `Pipeline([StandardScaler, LinearRegression()])` (OLS) — a tuned Ridge `α` grid produced no statistically clear winner either. OLS ships, by simplicity.

Throughout nb11 → nb15 these two pipelines are the **Week-2 reference models** — the linear baselines that every tree, forest, and boosting variant has to beat by a CI-clear margin to justify its added complexity. They appear as `reference_clf` and `reference_reg` in the setup cell and are the comparison floor in every model-comparison plot from here through nb14's selection ceremony.

---

## 1. Setup — Imports, Datasets, and Plot Helpers

The setup cell loads both datasets, defines the suffix convention (`_clf`, `_reg`), and registers two small plot helpers (`plot_train_val_curve`, `plot_predicted_vs_actual`) that every section will call. Locking the random seed to `RANDOM_SEED = 474` (course number) means every fold split, every tree fit, and every plot is reproducible across machines.

> 💡 **Gemini Prompt:** "Set up imports for sklearn DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, LogisticRegression, Ridge, cross_val_score, train_test_split, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing. Set RANDOM_SEED = 474 and define two helper functions: plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax) for overfitting diagnostics, and plot_predicted_vs_actual(y_true, y_pred, ax, title) for regression diagnostics."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474` printed back
> - [ ] Helpers `plot_train_val_curve` and `plot_predicted_vs_actual` defined and callable
> - [ ] No import errors
> - [ ] Display precision set to 4 digits


In [ ]:
# Setup — imports, seeds, display, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention (nb19 alignment) ---
CLF_COLOR = '#1f77b4'   # teal — classification accent
REG_COLOR = '#ff7f0e'   # orange — regression accent
GREY      = '#999999'

# --- Helper 1: train-vs-CV overfitting curve ---
def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    """Side-by-side train and CV-mean curves with CV std as error bars."""
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train (full training set)',
            linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s',
                label='5-fold CV mean ± SD', linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs)
    ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Helper 2: predicted-vs-actual scatter (the standard regression diagnostic) ---
def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual',
                             color=REG_COLOR):
    """Scatter of y_pred vs y_true with the y=x line of perfect prediction."""
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo = float(min(np.min(y_true), np.min(y_pred)))
    hi = float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Week-2 reference models (the linear baselines that survived nb09's CI-overlap test) ---
# Classification:  default LogReg(C=1.0) — random-search over C produced no CI-clear winner.
# Regression:      OLS                   — tuned Ridge α grid produced no CI-clear winner.
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Helpers defined: plot_train_val_curve, plot_predicted_vs_actual")
print(f"✓ Week-2 references defined: reference_clf (LogReg C=1.0), reference_reg (OLS)")
print(f"✓ Display precision: 4 digits")


**Reading the output:**

The setup cell does three jobs at once. First, it imports both `DecisionTreeClassifier` (for Wisconsin breast cancer) and `DecisionTreeRegressor` (for California housing) — the two algorithms we will pair-fit throughout the notebook. Second, it locks `RANDOM_SEED = 474` so every CV fold, every tree fit, and every plot reproduces identically across machines. Third, it defines two helper functions: `plot_train_val_curve` (the overfitting diagnostic you will see in Section 5) and `plot_predicted_vs_actual` (the standard regression-diagnostic scatter, mandatory for every regression model on the final poster).

---

## 2. Decision Tree Intuition — A Flowchart You Can Read

A decision tree is the analytics version of a flowchart. At every step it asks one yes/no question about a single feature — *"is annual income above \$60K?"*, *"is the lump's worst radius greater than 16.8?"* — and routes the observation left or right. After a handful of questions every observation arrives at a **leaf**, and the leaf holds the answer: a predicted class for classification, an average value for regression. That is the whole algorithm — no coefficients to interpret, no equations to memorize, just a chain of yes/no questions ending in a verdict.

A fitted tree comes in two equivalent views and you will see both side by side below. The **flowchart view** is the boxes-and-branches diagram you would draw on a whiteboard for a non-technical colleague. The **partition view** is the same tree's predictions painted onto the feature space — the right framing for comparing what a tree does to what a logistic regression or linear regression would do on the same data. Both views describe the same fitted model; they just emphasize different things.

The picture below uses a synthetic two-feature dataset so the geometry is visible on a flat plane. The real datasets in this notebook live in 8 and 30 dimensions, but the rule is identical — one feature per question, axis-aligned cuts in the data plane, and a single constant prediction inside each rectangle.

In [ ]:
# Section 2: build intuition with a small tree and the partition it creates.
# A two-feature synthetic dataset lets us see the tree's geometry on a flat plane.
X_toy, y_toy = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=2, class_sep=0.7, flip_y=0.05,
    random_state=RANDOM_SEED
)

# A depth=2 tree is small enough that every box and leaf is legible at a glance.
demo_tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_SEED).fit(X_toy, y_toy)

# Side-by-side: the flowchart view (left) and the partition it creates (right).
fig, (ax_tree, ax_plane) = plt.subplots(1, 2, figsize=(16, 6))

plot_tree(
    demo_tree,
    feature_names=['feature 1', 'feature 2'],
    class_names=['class A', 'class B'],
    filled=True,
    rounded=True,
    fontsize=11,
    ax=ax_tree,
)
ax_tree.set_title('Flowchart view — the fitted decision tree (max_depth=2)',
                  fontsize=13, fontweight='bold')

# Same tree, drawn on the 2D plane.
x_min, x_max = X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5
y_min, y_max = X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = demo_tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax_plane.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
ax_plane.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
                 edgecolor='k', s=40)
ax_plane.set_xlabel('feature 1')
ax_plane.set_ylabel('feature 2')
ax_plane.set_title('Partition view — the same tree drawn on the data',
                   fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Depth sweep: same dataset, deeper trees, finer partitions.
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, depth in zip(axes, [1, 2, 3, 6]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED).fit(X_toy, y_toy)
    Z = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
               edgecolor='k', s=40)
    ax.set_xlabel('feature 1')
    ax.set_ylabel('feature 2')
    ax.set_title(f'max_depth = {depth}  →  {t.get_n_leaves()} leaves',
                 fontsize=12, fontweight='bold')

fig.suptitle('Deeper trees carve the plane into more rectangles — at depth 6 the tree is already memorizing points',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✨ Each leaf in the flowchart on the left maps to one coloured rectangle on the right.")
print("✨ Every cut is horizontal or vertical — a tree never draws a diagonal line.")
print("✨ At depth 6 several leaves contain only one or two training points — that is overfitting in pictures.")


**Reading the output:**

Start with the flowchart on the left. The top box is the **root** — every training row enters here. The line inside the box (something like *"feature 2 <= 0.2"*) is the yes/no question this node asks. Rows that answer "yes" slide down the left branch; rows that answer "no" slide down the right. Each box also reports four pieces of information: `gini` is how mixed the classes are inside this box (0.0 means pure, 0.5 means a perfect 50/50 split), `samples` is how many training rows landed here, `value` is how those rows split between the two classes, and `class` is the box's current verdict. The boxes with no children — the bottom row — are the **leaves**, and once a row reaches a leaf the leaf's `class` is the tree's prediction.

Now look at the right panel. The plane is divided into four coloured rectangles, one per leaf in the flowchart. Pick any point in the plane, follow the questions in the tree, and you will land in the matching rectangle. That is what people mean when they call decision trees *interpretable*: the path from input to prediction is a short list of yes/no questions you could read aloud to a loan officer, a clinician, or a hiring manager and they would understand the decision without ever opening the model.

The four-panel figure below sweeps the depth lever on the same dataset. At `max_depth = 1` the tree asks one question, the plane is cut once, and many points sit on the wrong side of the line — that is **underfitting**. By `max_depth = 6` the tree has carved out so many tiny rectangles that some leaves contain only one or two training points, drawn around them like little fences — that is **overfitting**. Somewhere in the middle is a sweet spot, and finding it on real data is exactly what Sections 6 and 7 will quantify.

> **A question that often comes up here:** *"why are all the boundaries horizontal or vertical?"* Because each split asks about exactly one feature at a time, the cut it draws has to be perpendicular to that feature's axis. A logistic regression can draw a single diagonal line that uses both features at once; a tree has to approximate the same diagonal with a staircase of horizontal and vertical cuts. Trees can get there, but every step of the staircase costs depth — and depth is what makes a tree variance-hungry on small datasets, as the depth sweep on the right already hints at.

**Key takeaway:** A tree's depth is its bias-variance dial. Shallow trees ask too few questions and miss real signal (high bias). Deep trees ask so many questions that each leaf holds only a handful of training rows and the model starts memorizing them (high variance). The rest of this notebook is about finding the depth that balances the two on real data — not on this toy plane.

---

## 3. Load Both Datasets — Three Holdout Splits, Two Locked Envelopes

Time to set up the actual data. Both projects follow the same rule the course locked in back in nb01: split the rows once into **60% training / 20% validation / 20% test**, then leave the test envelope sealed until nb14. Every model fit, every diagnostic, every tuning decision between here and nb14 lives on the **training set** and is evaluated through 5-fold cross-validation. The **validation set** is your private held-out sample — available for any one-shot check you cannot run through CV — and the **test set** is the final blind exam the auditor grades once.

`random_state=RANDOM_SEED` keeps the split identical across runs. Inside the training set, evaluation goes through **5-fold cross-validation**, the procedure you locked in during nb08. The two tracks use slightly different CV splitters because the data shapes are different. Classification uses `StratifiedKFold`, which keeps each fold's mix of malignant and benign biopsies close to the overall \~63% benign / \~37% malignant balance — important whenever one class is the minority. Regression uses plain `KFold` because there are no classes to preserve; the housing target is a continuous price, and any random fold is already representative.

The Wisconsin Breast Cancer data is small (569 patients, 30 features per biopsy) and California Housing is large (20,640 census tracts, 8 features per tract). That roughly 36× size gap will matter in the next sections — small datasets let trees overfit fast, large datasets tolerate deeper trees but slow CV down. Both datasets reappear in every comparison from here through Section 7.

> 💡 **Gemini Prompt:** "Apply the course's 60/20/20 split twice. For the breast-cancer dataset: first carve off 20% as `X_test_clf` / `y_test_clf` with `test_size=0.20`, `random_state=474`, `stratify=y_clf`; then split the remaining 80% with `test_size=0.25` to produce `X_train_clf` / `X_val_clf`. Repeat for California Housing without stratification to produce `X_train_reg` / `X_val_reg` / `X_test_reg`. Create `cv_clf = StratifiedKFold(5)` and `cv_reg = KFold(5)`. Print sizes of all three splits per spine and remind the reader that the test sets stay LOCKED until nb14."
>
> **After running, verify:**
> - [ ] Classification: train \~341, val \~114, test \~114 (out of 569)
> - [ ] Regression: train \~12,384, val \~4,128, test \~4,128 (out of 20,640)
> - [ ] All splits use `random_state=RANDOM_SEED`
> - [ ] Both `X_test_*` printouts include the LOCKED marker


In [ ]:
# --- Classification track: Wisconsin breast cancer (State Health Department) ---
data_clf = load_breast_cancer(as_frame=True)
X_clf = data_clf.data
y_clf = data_clf.target

# 60/20/20 split: first peel off 20% test, then split the remaining 80% into 75/25 → 60/20.
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing (HomeValue Analytics) ---
data_reg = fetch_california_housing(as_frame=True)
X_reg = data_reg.data
y_reg = data_reg.target  # target is median house value in units of USD 100K

X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (State Health Department / Wisconsin Breast Cancer) ===")
print(f"  Features:  {X_clf.shape[1]}")
print(f"  Train:     {len(X_train_clf):>5} patients   (used for all CV from here through Section 7)")
print(f"  Val:       {len(X_val_clf):>5} patients   (reserved for any one-shot held-out check)")
print(f"  Test:      {len(X_test_clf):>5} patients   [LOCKED until nb14]")
print(f"  Class balance (train): {y_train_clf.mean():.3f} positive (benign)")

print()
print("=== REGRESSION (HomeValue Analytics / California Housing) ===")
print(f"  Features:  {X_reg.shape[1]}")
print(f"  Train:     {len(X_train_reg):>5} tracts     (used for all CV from here through Section 7)")
print(f"  Val:       {len(X_val_reg):>5} tracts     (reserved for any one-shot held-out check)")
print(f"  Test:      {len(X_test_reg):>5} tracts     [LOCKED until nb14]")
print(f"  Target:    median house value, USD 100K units")
print(f"  y_train range: USD {y_train_reg.min()*100:.0f}K – USD {y_train_reg.max()*100:.0f}K, "
      f"mean USD {y_train_reg.mean()*100:.0f}K")


**Reading the output:**

Three splits per spine, two locked envelopes, two CV plans. Notice the size gap. The Wisconsin biopsy data hands you about 341 patients to train on; California Housing hands you about 12,384 tracts. The housing set is roughly **36× larger**, and that asymmetry will shape every model decision in this notebook. With only 341 patients and 30 features, a classification tree can memorize individual biopsies after just a handful of splits — small data, sharp risk of overfitting. The regression tree has more than twelve thousand tracts to learn from before that danger kicks in, so it tolerates deeper trees, but its CV runs take noticeably longer (which is why a few regression cells later in the notebook will drop to 3-fold CV when 5-fold would not finish in a reasonable time on Colab).

The class-balance line on the classification side is worth a second look: about 63% benign, 37% malignant in the training set. That is the balance `StratifiedKFold` preserves in every CV fold. Without stratification, a random fold could land mostly benign cases and the model's accuracy would look misleadingly stable; with stratification, each fold sees the same realistic class mix that the State Health Department's clinic actually faces day to day.

> **A question that often comes up here:** *"what is the validation set for, if every diagnostic in this notebook uses CV on the training set?"* In nb11 specifically, `X_val_clf` and `X_val_reg` sit on the sidelines — every model-comparison plot you see in Sections 4–7 is built from cross-validation on the training set. The validation set becomes useful when you need a **one-shot** held-out check that does not naturally fit a CV loop, such as the calibration plot you will see in nb16 or any peek-once threshold-sweep diagnostic. The test set is different — it stays sealed until nb14's selection ceremony. Treat `X_test_clf`, `y_test_clf`, `X_test_reg`, and `y_test_reg` like sealed envelopes in the State Health Department's audit room: they exist, they have your name on them, and you open them exactly once.

**Key takeaway:** Two cases, two namespaces, 60/20/20 splits, two sealed envelopes. Everything from here through Section 7 evaluates on the **training** sets via CV — `X_val_*` is reserved for one-shot checks later in the course, and `X_test_*` stays closed until nb14.

---

## 4. Classification Tree Example — Wisconsin Breast Cancer

Time to run the algorithm on the real classification problem. The fit below trains a `DecisionTreeClassifier` with `max_depth=3` on the breast-cancer training set — three layers of yes/no questions, no more. That depth is shallow on purpose. Three splits is plenty for a strong first baseline on this dataset, and it is small enough that `plot_tree` produces a flowchart a clinician can actually read on a single page.

Three numbers tell you how the fit went: the **training accuracy** (how well the tree memorizes the 341 patients it saw during fitting), the **5-fold CV accuracy** (how well the same algorithm generalizes when trained and scored on different folds), and the **gap** between them. A large gap is the early-warning siren for overfitting. We will also visualize the tree itself, because the headline value of trees in a regulated setting is not just *"the model got X% accuracy"* — it is *"here is the four-question rule the model used, and a pathologist can sanity-check every split against the clinical literature."*

> 💡 **Gemini Prompt:** "Fit a DecisionTreeClassifier(max_depth=3, random_state=474) on X_train_clf, y_train_clf. Print training accuracy on the full training set and 5-fold CV accuracy mean ± SD using cv_clf and scoring='accuracy'. Then call plot_tree with feature_names, class_names, filled=True."
>
> **After running, verify:**
> - [ ] Train accuracy in the 0.96–0.99 range
> - [ ] CV mean ± SD reported with 4-digit precision
> - [ ] Train-CV gap printed explicitly
> - [ ] Tree visualization shows ≤ 8 leaves


In [ ]:
# Classification tree at max_depth=3 — fit, score, visualize.
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_clf.fit(X_train_clf, y_train_clf)

train_acc_clf = tree_clf.score(X_train_clf, y_train_clf)
cv_scores_clf = cross_val_score(tree_clf, X_train_clf, y_train_clf,
                                cv=cv_clf, scoring='accuracy')
cv_mean_clf = cv_scores_clf.mean()
cv_std_clf  = cv_scores_clf.std(ddof=1)

print("=== CLASSIFICATION TREE (max_depth=3) ===")
print(f"Train accuracy:        {train_acc_clf:.4f}")
print(f"5-fold CV accuracy:    {cv_mean_clf:.4f}  (SD = {cv_std_clf:.4f})")
print(f"Overfit gap (train−CV): {train_acc_clf - cv_mean_clf:.4f}")
print(f"Tree leaves:           {tree_clf.get_n_leaves()}")

# --- Visualize the tree ---
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    tree_clf,
    feature_names=X_clf.columns,
    class_names=data_clf.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title('Wisconsin Breast Cancer — Classification Tree at max_depth=3',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Each leaf shows: gini impurity, sample count, class distribution, predicted class.")
print("💡 Darker shading = purer leaf (mostly one class).")


**Reading the output:**

A `max_depth=3` tree on the breast-cancer training set typically lands a 5-fold CV accuracy in the 0.93–0.95 range with a train–CV gap of 0.02–0.04 — a healthy sign that the tree is learning patterns that generalize, not just memorizing the 341 patients in the training set. The tree itself uses only seven or eight leaves; you can read every decision path in plain English, and that is exactly what makes it deployable in a regulated clinical workflow. The root split almost always lands on `worst radius` or `worst perimeter`, both of which are well-established malignancy markers in the clinical literature. That is the kind of plausibility check that builds trust with the State Health Department's review board when they audit the model.

> **A question that often comes up here:** *"what does the gini number on each node mean?"* Gini impurity is `1 − Σ p_i²` where `p_i` is the fraction of samples in class `i`. A pure leaf (all malignant or all benign) has gini = 0; a perfectly mixed 50/50 leaf has gini = 0.5. The tree's split criterion at every internal node is the same simple rule: *pick the feature and threshold that drop the weighted-average gini of the two children the most*. That single objective — minimize child impurity — is the entirety of how a classification tree decides where to cut.

**Key takeaway:** A shallow classification tree is interpretable, plausible, and already a competitive baseline — but the 2–4 point train–CV gap suggests the tree could be tuned for more depth. Section 6 will sweep depth on both spines to make the bias-variance trade-off visible, and Exercise 1 will let you pick the right depth using the one-standard-error rule.

---

## 5. Regression Tree Example — California Housing

The regression tree on California Housing uses the same machinery you just saw on the breast-cancer data — recursive binary splits, axis-aligned cuts, a constant prediction per leaf — with two natural adaptations for a continuous target. First, the **split criterion** switches from gini impurity to **mean squared error**: the tree now picks the feature and threshold that drop the weighted-average MSE of the two children the most. Same logic, different metric. Second, the **leaf prediction** is no longer a majority-class vote; it is the **average house value** of all training tracts that landed in that leaf. A leaf containing 47 tracts whose median values average USD 240K predicts USD 240K for any new property that follows the same path through the tree.

The visualization below packs three diagnostics into a single figure. The top panel shows the fitted tree itself, the same way you saw the classification tree in the previous section. The bottom-left panel is the **predicted-vs-actual scatter** — the mandatory diagnostic for any regression model. The bottom-right panel slides `MedInc` (median income) across its full training range while holding every other feature at its training median; that traces a one-dimensional cross-section through the tree's prediction surface and produces the **step function** that is the visual signature of every regression tree.

> 💡 **Gemini Prompt:** "Fit DecisionTreeRegressor(max_depth=3, random_state=474) on X_train_reg, y_train_reg. Print train R², 5-fold CV R² mean ± SD using cv_reg, and train RMSE in dollars. Plot three panels: plot_tree, predicted-vs-actual scatter (use the helper), and a step-function showing tree.predict over MedInc (with other features held at their training median). Use cross_val_score with scoring='neg_root_mean_squared_error' for the RMSE CV."
>
> **After running, verify:**
> - [ ] Train R² in 0.50–0.70 range; CV R² slightly lower
> - [ ] CV RMSE reported in USD (multiply by 100,000)
> - [ ] Step-function plot shows 8 distinct horizontal levels (one per leaf)
> - [ ] Predicted-vs-actual scatter has the y=x reference line drawn


In [ ]:
# Regression tree at max_depth=3 — fit, score, visualize, step-function.
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_SEED)
tree_reg.fit(X_train_reg, y_train_reg)

train_r2_reg  = tree_reg.score(X_train_reg, y_train_reg)
cv_r2_reg     = cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                cv=cv_reg, scoring='r2')
cv_rmse_reg   = -cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                 cv=cv_reg, scoring='neg_root_mean_squared_error')

print("=== REGRESSION TREE (max_depth=3) ===")
print(f"Train R²:               {train_r2_reg:.4f}")
print(f"5-fold CV R²:           {cv_r2_reg.mean():.4f}  (SD = {cv_r2_reg.std(ddof=1):.4f})")
print(f"5-fold CV RMSE:         {cv_rmse_reg.mean():.4f} (in 100K USD units)")
print(f"5-fold CV RMSE in USD:  USD {cv_rmse_reg.mean()*100_000:,.0f}")
print(f"Tree leaves:            {tree_reg.get_n_leaves()}")

# --- Three-panel diagnostic ---
fig = plt.figure(figsize=(20, 11))
gs  = fig.add_gridspec(2, 2, height_ratios=[1.4, 1])

# Top: the regression tree itself
ax_tree = fig.add_subplot(gs[0, :])
plot_tree(
    tree_reg,
    feature_names=X_reg.columns,
    filled=True, rounded=True, fontsize=10, ax=ax_tree,
)
ax_tree.set_title('California Housing — Regression Tree at max_depth=3',
                  fontsize=14, fontweight='bold')

# Bottom-left: predicted-vs-actual on training set
ax_pva = fig.add_subplot(gs[1, 0])
y_pred_reg = tree_reg.predict(X_train_reg)
plot_predicted_vs_actual(y_train_reg, y_pred_reg, ax_pva,
                         title='Predicted vs Actual (training set, USD 100K units)')

# Bottom-right: 1D step function — predict as MedInc varies, others at median
ax_step = fig.add_subplot(gs[1, 1])
medinc_range = np.linspace(X_train_reg['MedInc'].min(),
                           X_train_reg['MedInc'].max(), 500)
median_row = X_train_reg.median()
grid = pd.DataFrame([median_row] * 500, columns=X_train_reg.columns)
grid['MedInc'] = medinc_range
y_step = tree_reg.predict(grid)

ax_step.scatter(X_train_reg['MedInc'], y_train_reg,
                alpha=0.10, s=6, color=GREY, label='Training data')
ax_step.plot(medinc_range, y_step, color=REG_COLOR, linewidth=2.5,
             label='Tree prediction (other features at median)')
ax_step.set_xlabel('MedInc (median income, 10K USD units)')
ax_step.set_ylabel('Predicted house value (USD 100K units)')
ax_step.set_title('Tree predicts step-functions, not smooth curves',
                  fontsize=12, fontweight='bold')
ax_step.legend()
ax_step.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 The step function has exactly as many levels as the tree has leaves.")
print("💡 This is the visual signature of axis-aligned splits — no smoothness.")


**Reading the output:**

A `max_depth=3` regression tree on California Housing typically lands a 5-fold CV R² around 0.50 and a CV RMSE near USD 80K. That is nowhere near state-of-the-art on this benchmark — Random Forests (nb12) and Gradient Boosting (nb13) will roughly halve the RMSE — but it already crushes the *"predict the training mean"* baseline that HomeValue Analytics' pricing analysts had been doing by hand before this engagement. As a first non-linear model on the regression spine, it is a meaningful improvement and a defensible starting point.

The eight leaves at the bottom of the top panel are the **entire prediction repertoire** of this tree. Every leaf reports a `value` line — the **mean house value** of every training tract that followed that leaf's specific path of yes/no answers, and exactly the price the tree quotes for any new property that traces the same path. The eight leaf-means span roughly **USD 127K** at the cheapest end (low `MedInc`, high `AveOccup` — small homes packed with occupants) to **USD 482K** at the most expensive (`MedInc` above 9.07 — the wealthiest tracts in training). The split structure tells the rest of the story: tracts with `MedInc ≤ 5.59` (median household income up to USD 56K) land in four leaves between USD 127K and USD 285K; tracts with `MedInc > 5.59` land in four leaves between USD 295K and USD 482K. A `max_depth=3` tree can quote exactly **eight** prices for any property you hand it — and those same eight numbers reappear as the eight horizontal levels in the step-function panel on the bottom-right, which is nothing more than the leaf-means plotted along the `MedInc` axis with everything else held at its training median.

The step-function plot in the bottom-right panel is the visual signature of every regression tree. Across the entire range of `MedInc`, the prediction takes only as many distinct values as the tree has leaves — eight in this case. Inside each step the tree is constant; at the step boundaries it jumps. This has one important consequence: a regression tree **cannot extrapolate**. If a new property has a `MedInc` higher than anything seen in training, the tree clamps to the highest-leaf average — it has no notion of *"the price keeps rising past USD 500K."* Linear models extrapolate cleanly along a regression line; trees do not. The trade you are making is interpretability and non-linearity in exchange for boundedness and step-function predictions, and that trade is the central characteristic of the algorithm.

> **A question that often comes up here:** *"if the tree only predicts step values, isn't that always worse than a linear model?"* Not always. When the true relationship is **non-monotonic** — for example, when neighbourhood effects cause house prices to drop in certain income brackets — the tree captures that bend, and a linear model cannot. When the relationship is **monotonic and smooth**, the linear model wins. Section 7 makes that comparison concrete on both spines using the Week-2 reference pipelines from nb09.

**Key takeaway:** Regression trees buy you interpretability and non-linearity at the cost of step-function predictions and zero extrapolation. Whether that trade is worth making depends on the data — which is exactly what the next two sections quantify on the two real cases.

---

## 6. The Overfitting Problem — Paired Across Both Cases

Sections 4 and 5 each fit a single tree at `max_depth=3`. The obvious next question is: *what would happen at depth 1, or depth 5, or with no depth cap at all?* The cell below answers that on **both spines at the same time** by sweeping `max_depth ∈ {1, 2, 3, 5, 10, 20, None}` and recording the training score and the 5-fold CV score at every setting. The result is a two-panel plot — classification on the left, regression on the right — that puts the bias-variance trade-off in front of you in a single picture.

The number to watch is not the training score on its own, and not the CV score on its own — it is the **gap** between them. A shallow tree under-asks: training and CV are both mediocre, and the gap is small (high bias). A deep tree memorizes: training climbs toward 1.0 but CV plateaus or even degrades, and the gap explodes (high variance). The right depth is wherever the CV score is highest and the gap is still respectable. The plateau on each spine lands in a different place because the two datasets differ in size, but the shape of the story is identical on both.

> 💡 **Gemini Prompt:** "Sweep max_depth in [1, 2, 3, 5, 10, 20, None] for both DecisionTreeClassifier on X_train_clf and DecisionTreeRegressor on X_train_reg. For each depth and each case, fit on full training set, record train score and 5-fold CV mean ± SD. Build a 1×2 subplot using the plot_train_val_curve helper — left panel classification (accuracy), right panel regression (R²). Title the figure to make the universality of the overfitting pattern explicit."
>
> **After running, verify:**
> - [ ] Both panels share the same x-axis layout (same depth values)
> - [ ] Classification curve plateaus near 0.95+ around depth 3–5
> - [ ] Regression curve plateaus around depth 8–10
> - [ ] Train curves both reach 1.0 at unrestricted depth (None)


In [ ]:
# Paired depth sweep — same pattern emerges on both cases.
depths = [1, 2, 3, 5, 10, 20, None]

# --- Classification sweep ---
clf_train, clf_val_mean, clf_val_std = [], [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_SEED).fit(X_train_clf, y_train_clf)
    clf_train.append(t.score(X_train_clf, y_train_clf))
    s = cross_val_score(t, X_train_clf, y_train_clf, cv=cv_clf, scoring='accuracy')
    clf_val_mean.append(s.mean())
    clf_val_std.append(s.std(ddof=1))

# --- Regression sweep ---
reg_train, reg_val_mean, reg_val_std = [], [], []
for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_SEED).fit(X_train_reg, y_train_reg)
    reg_train.append(t.score(X_train_reg, y_train_reg))
    s = cross_val_score(t, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2')
    reg_val_mean.append(s.mean())
    reg_val_std.append(s.std(ddof=1))

# --- Side-by-side panels ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_train_val_curve(
    [str(d) for d in depths], clf_train, clf_val_mean, clf_val_std,
    xlabel='max_depth', ylabel='Accuracy',
    title='Classification — Wisconsin Breast Cancer',
    ax=axes[0], color_val=CLF_COLOR
)
plot_train_val_curve(
    [str(d) for d in depths], reg_train, reg_val_mean, reg_val_std,
    xlabel='max_depth', ylabel='R²',
    title='Regression — California Housing',
    ax=axes[1], color_val=REG_COLOR
)

fig.suptitle('Same overfitting pattern on both spines — train rises to 1.0; CV plateaus and then degrades',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print the train–CV gap table
print("=== TRAIN − CV GAP BY DEPTH ===")
print(f"{'depth':>8} | {'clf train':>10} | {'clf CV':>10} | {'gap':>8} ||"
      f" {'reg train':>10} | {'reg CV':>10} | {'gap':>8}")
for i, d in enumerate(depths):
    print(f"{str(d):>8} | {clf_train[i]:>10.4f} | {clf_val_mean[i]:>10.4f} | "
          f"{clf_train[i]-clf_val_mean[i]:>8.4f} || "
          f"{reg_train[i]:>10.4f} | {reg_val_mean[i]:>10.4f} | "
          f"{reg_train[i]-reg_val_mean[i]:>8.4f}")


**Reading the output:**

The two panels tell the same story at different scales. On the left, classification accuracy rises sharply from depth 1 to depth 3 and then plateaus near 0.93. On the right, regression R² climbs from about 0.30 at depth 1 to a plateau near 0.65 at depth 8–10. In **both** cases the training score keeps marching toward 1.0 long after the CV score has stopped improving — that growing gap is the pure-overfitting signature, and it is the same shape on both spines despite the very different dataset sizes.

Notice where the plateau lands on each side. Classification levels off around depth 3–5 because the dataset is small — only 30 features and \~341 training patients, so the tree runs out of room to grow before it starts memorizing. Regression plateaus much later, around depth 8–10, because there are over 12,000 training tracts to learn from. The plateau depth is roughly the depth at which the leaves get too small to be statistically reliable, and that threshold depends on dataset size, not on the algorithm itself. The same lesson lands in both panels: bigger training sets tolerate deeper trees.

Reading the table for **MedScreen's** classification spine, CV accuracy peaks at depth 5 (0.935 ± 0.029) and the runner-up is depth 3 at 0.932 with a smaller train–CV gap (0.047 vs 0.059). Both depths sit within one SD of the peak, so the **parsimonious pick is depth 3** — statistically tied with depth 5, healthier gap, and three yes/no questions a State Health Department oncologist can read off a flowchart in seconds. Anything past depth 10 holds train at 1.0 while CV slips to 0.918 and the gap doubles to 0.082 — textbook overfitting, and the depth band MedScreen should never deploy.

For **HomeValue Analytics'** regression spine the picture is different because the dataset is 36× larger. CV R² rises smoothly from 0.30 at depth 1 to **0.613 at depth 5** with a **tight train–CV gap of 0.02**. Push to depth 10 and CV climbs to 0.678 — a six-and-a-half R² point bump — but the gap balloons to 0.18, and at depth 20 the gap reaches 0.39 while CV slides back to 0.607. That is the textbook overfitting signature the section opened against: training climbs toward 1.0 while CV plateaus, and the lift in CV is bought on credit. **Depth 5 is the recommendation**: the deepest depth where the gap is still respectable, capturing the bulk of the non-linearity HomeValue's pricing committee actually wants — the extra R² at depth 10 is not earned signal, it is fold-specific noise that will not survive the test set in nb14. Section 7 hands this depth-5 tree to the head-to-head against the Week-2 OLS reference.

> **A question that often comes up here:** *"why does CV sometimes degrade after the plateau and sometimes stay flat?"* When CV degrades, it is because the deep tree has carved out tiny leaves that fit noise specific to the training set, and on the held-out folds those leaves predict the wrong class or the wrong leaf-mean. When CV stays flat after the plateau, the tree is still overfitting — the overfitting is just symmetric across folds, so the average does not move much. Either way, the **train–CV gap** is the more reliable diagnostic: it grows monotonically with depth on every dataset.

**Key takeaway:** The overfitting pattern is universal — same shape, different scale. Train climbs toward 1.0, CV plateaus, the gap explodes. From this seven-point sweep MedScreen picks **depth 3** (parsimonious + healthy gap) and HomeValue picks **depth 5** (deepest depth where the train–CV gap stays respectable; depth 10 raises CV by 6.5 R² points but blows the gap from 0.02 to 0.18 — overfitting credit, not earned signal). Exercises 1 and 2 reconfirm these picks on a finer depth grid using the one-standard-error rule; Section 7 then asks whether either tree has earned the right to displace the Week-2 linear reference.

---

## 7. Tree vs Week-2 Reference — Paired

Time for the head-to-head. Each tree is compared against the **Week-2 reference model** for its track — the linear baseline that survived nb09's CI-overlap test. These are not arbitrary linear analogues; they are the exact pipelines your final-project group's M2 baseline (or its sibling) shipped at the end of last week:

- **Classification:** `DecisionTreeClassifier(max_depth=3)` vs `reference_clf` = `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])`. Primary metric: ROC-AUC.
- **Regression:** `DecisionTreeRegressor(max_depth=5)` vs `reference_reg` = `Pipeline([StandardScaler, LinearRegression()])` (OLS). Primary metric: R².

The output is two side-by-side bar charts with CV mean and **Student's *t* 95% CIs** shown as error bars — the same construction from nb08: `half_w = t_{0.975, k-1} × sd / √k`, with `t_{0.975, 4} ≈ 2.776` at `k=5`. The CI-overlap rule from nb08 and nb09 still applies — if the tree's 95% CI overlaps the reference's, the two models are in a statistical tie and the **simpler model wins by default**. The tree only earns the right to displace the linear baseline when its CV interval clears the reference's by a visible margin. That standard is deliberately unforgiving: every new model in the candidate roster from here through nb14 has to clear the same bar.

> 💡 **Gemini Prompt:** "Compare DecisionTreeClassifier(max_depth=3) against reference_clf (the Week-2 LogReg pipeline) on the breast cancer training set using cv_clf and scoring='roc_auc'. Compare DecisionTreeRegressor(max_depth=5) against reference_reg (the Week-2 OLS pipeline) on the California housing training set using cv_reg and scoring='r2'. For every model compute mean, sd (ddof=1), half_w = t_crit × sd / √k with t_crit = scipy.stats.t.ppf(0.975, df=k-1) at k=5, ci_low and ci_high — print one DataFrame per spine showing all five columns. Two side-by-side bar charts with CV mean and 95% CI (half_w) error bars; label the reference bars 'Week-2 reference' and the tree bars 'Decision Tree'. Then print a CI-overlap verdict per spine."
>
> **After running, verify:**
> - [ ] DataFrames carry `mean`, `sd`, `half_w`, `ci_low`, `ci_high` for every model
> - [ ] Error bars use the 95% CI half-width (`t_crit × sd / √k`), not raw SD
> - [ ] CI-overlap verdict printed for both spines
> - [ ] Reference bars carry the label "Week-2 reference: LogReg(C=1.0)" / "Week-2 reference: OLS"


In [ ]:
# Tree vs Week-2 reference — paired comparison with 5-fold CV mean and 95% CIs (Student's t, df=k-1).
clf_models = {
    'Decision Tree (depth=3)':              DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Week-2 reference: LogReg(C=1.0)':      reference_clf,
}
reg_models = {
    'Decision Tree (depth=5)':              DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'Week-2 reference: OLS':                reference_reg,
}

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)  # ≈ 2.776 — the same Student's t constant from nb08

def summarize_cv(model, X, y, cv, scoring):
    s = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    mean = s.mean()
    sd = s.std(ddof=1)
    half_w = t_crit * sd / np.sqrt(k)
    return {'mean': mean, 'sd': sd, 'half_w': half_w,
            'ci_low': mean - half_w, 'ci_high': mean + half_w}

clf_results = [{'model': name, **summarize_cv(m, X_train_clf, y_train_clf, cv_clf, 'roc_auc')}
               for name, m in clf_models.items()]
reg_results = [{'model': name, **summarize_cv(m, X_train_reg, y_train_reg, cv_reg, 'r2')}
               for name, m in reg_models.items()]

clf_df = pd.DataFrame(clf_results)
reg_df = pd.DataFrame(reg_results)

cols = ['model', 'mean', 'sd', 'half_w', 'ci_low', 'ci_high']
print("=== CLASSIFICATION (5-fold CV ROC-AUC, mean ± 95% CI) ===")
print(clf_df[cols].to_string(index=False))
print()
print("=== REGRESSION (5-fold CV R², mean ± 95% CI) ===")
print(reg_df[cols].to_string(index=False))

# --- CI-overlap verdict per spine (with dominance tiebreaker) ---
def ci_overlap_verdict(df, contender_name, reference_name):
    contender = df[df['model'] == contender_name].iloc[0]
    reference = df[df['model'] == reference_name].iloc[0]
    overlap = not (contender['ci_high'] < reference['ci_low']
                   or reference['ci_high'] < contender['ci_low'])
    if not overlap:
        winner = contender['model'] if contender['mean'] > reference['mean'] else reference['model']
        return f"CIs do NOT overlap → {winner} wins outright (CI-clear margin)"
    # CIs overlap: apply the dominance tiebreaker — top-by-mean keeps the win
    # when its CV CI lower bound is also above the runner-up's CI lower bound.
    if contender['mean'] > reference['mean']:
        if contender['ci_low'] > reference['ci_low']:
            return (f"CIs overlap → dominance tiebreaker → {contender['model']} wins "
                    f"as the modest winner (higher mean AND higher CI lower bound)")
        return (f"CIs overlap and reference's CI lower bound is not exceeded → "
                f"statistical tie → simpler model wins ({reference['model']})")
    return f"CIs overlap and reference's mean is higher → {reference['model']} wins on both fronts"

print()
print("Classification verdict:", ci_overlap_verdict(
    clf_df, 'Decision Tree (depth=3)', 'Week-2 reference: LogReg(C=1.0)'))
print("Regression verdict:    ", ci_overlap_verdict(
    reg_df, 'Decision Tree (depth=5)', 'Week-2 reference: OLS'))

# --- Side-by-side bar plots with 95% CI error bars ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
ax.bar(clf_df['model'], clf_df['mean'], yerr=clf_df['half_w'], capsize=10,
       color=[CLF_COLOR, GREY], edgecolor='black')
for i, row in clf_df.iterrows():
    ax.text(i, row['mean'] + row['half_w'] + 0.005, f"{row['mean']:.4f}", ha='center', fontsize=10)
ax.set_ylabel('5-fold CV ROC-AUC (mean ± 95% CI)')
ax.set_title('Classification — Tree vs Week-2 reference', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(reg_df['model'], reg_df['mean'], yerr=reg_df['half_w'], capsize=10,
       color=[REG_COLOR, GREY], edgecolor='black')
for i, row in reg_df.iterrows():
    ax.text(i, row['mean'] + row['half_w'] + 0.005, f"{row['mean']:.4f}", ha='center', fontsize=10)
ax.set_ylabel('5-fold CV R² (mean ± 95% CI)')
ax.set_title('Regression — Tree vs Week-2 reference', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle("Tree vs Week-2 reference — 5-fold CV mean ± Student's t 95% CI",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On the breast-cancer side, the **classification tree at depth 3** lands a 5-fold CV ROC-AUC of **0.922** with a 95% CI of **[0.864, 0.981]** — a wide interval driven by a fold-to-fold SD of 0.047 on only \~341 training patients. The Week-2 reference (`StandardScaler + LogisticRegression(C=1.0)`) lands at **0.994** with a much tighter 95% CI of **[0.986, 1.001]** (the upper bound runs slightly past 1.0 because Student's *t* does not know ROC-AUC is bounded; the practical interpretation is *"near-ceiling, with very low fold-to-fold noise"*). The tree's CI upper bound (0.981) sits **below** the reference's CI lower bound (0.986) — the bars do **NOT** overlap. **LogReg wins outright by a CI-clear margin**, exactly the pattern you would expect on a dataset that is essentially linearly separable in standardized cell-nucleus space. **MedScreen's pick: keep `LogisticRegression(C=1.0)`** — the tree is interpretable but it has not earned displacement.

On the California-housing side, the **regression tree at depth 5** lands a 5-fold CV R² of **0.613** with a tight 95% CI of **[0.604, 0.621]** — narrow, reflecting low fold-to-fold variance on 12,384 training tracts. The Week-2 reference (`StandardScaler + OLS`) lands at **0.586** with a wider 95% CI of **[0.554, 0.619]**. The tree wins on mean by 2.6 R² points, and the two CIs do overlap on the upper end in the band **[0.604, 0.619]** — but the **CV CI lower bound is where the tie breaks**: the tree's lower bound (**0.604**) sits *above* OLS's lower bound (**0.554**) by a full **5 R² points**. Even at its worst-case fold-to-fold draw, the tree outperforms OLS's worst-case by a wide margin. The tree dominates on both **mean** and **worst-case CI bound** — by the **dominance tiebreaker** (top-by-mean keeps the win when its CV CI lower bound is *above* the runner-up's), the upper-end CI overlap does not earn a strict tie. **HomeValue's pick: ship `DecisionTreeRegressor(max_depth=5)`** as a **modest winner** over OLS. The tree captures non-linearity and interaction structure OLS cannot, and the lower-CI dominance formalizes that as a defensible verdict. OLS stays on as the close-second runner-up — and nb12 (random forests) and nb13 (gradient boosting) will widen the gap further on this same data.

> **A question that often comes up here:** *"on classification the Week-2 LogReg still wins — why bother with trees at all?"* Three answers. First, today's single tree is the floor for the next two notebooks. Random Forests (nb12) and Gradient Boosting (nb13) are ensembles **of trees**, not ensembles of linear models, and they routinely lift CV scores well past whichever baseline holds per case. On regression today's tree has already cleared OLS under the dominance tiebreaker, and nb12-nb13 will widen that gap; on classification the linear reference still holds, and the ensembles will have to earn their displacement. Second, trees give you a kind of **interpretability** the linear baseline simply cannot match — a State Health Department clinician can read three yes/no questions off a flowchart, but no one is reading 30 standardized logistic coefficients in their head. Third, the comparison itself is the discipline: benchmarking every new model against a CV-stable reference is exactly what lets nb14's selection ceremony name a defensible champion across five candidates per spine.

**Key takeaway:** Two stakeholders, two verdicts. **MedScreen keeps `LogisticRegression(C=1.0)`** — the classification tree at depth 3 (CV ROC-AUC 0.922) does not clear the linear reference (0.994) under the CI-overlap rule. **HomeValue switches to `DecisionTreeRegressor(max_depth=5)`** — the regression tree (CV R² 0.613, lower-CI 0.604) dominates OLS (0.586, lower-CI 0.554) on both mean and worst-case CI bound under the dominance tiebreaker, even with the upper-end CI overlap. The single tree's value carries through to the rest of the course in two ways: as the regression baseline that already cleared the Week-2 reference on its own (the new floor nb12-nb13's ensembles will have to clear), and as a building block for the ensembles in nb12 and nb13 that will widen the gap further on California Housing.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification Tree's Depth

**Task:** Use 5-fold cross-validation to pick the optimal `max_depth` for `DecisionTreeClassifier` on the Wisconsin breast cancer training set, scoring on **ROC-AUC**.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 4, 5, 6, 7, 8, 10, 15]`.
2. For each depth, compute the 5-fold CV mean and SD using `cv_clf`.
3. Plot CV mean with SD error bars across depths.
4. Report the depth with the highest CV mean **and** the simplest depth whose CV mean is within one SD of the best (the **one-standard-error rule**).
5. Write 3 short findings: what does the curve tell you about the bias-variance trade-off?

---

> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeClassifier with random_state=474 over depths [2,3,4,5,6,7,8,10,15] on X_train_clf, y_train_clf using cv_clf and scoring='roc_auc'. Report mean ± SD for each depth in a DataFrame, plot mean with SD error bars, mark the highest-mean depth with a dashed line, and apply the one-standard-error rule to pick the simplest competitive depth."
>
> **After running, verify:**
> - [ ] DataFrame has 9 rows (one per depth)
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth and one-SE-rule depth both reported
> - [ ] CV ROC-AUC values are in 0.92–0.99 range


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune classification-tree depth using 5-fold CV ROC-AUC.
# Apply the one-standard-error rule when selecting the final depth.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression Tree's Depth

**Task:** Use 5-fold cross-validation to pick the optimal `max_depth` for `DecisionTreeRegressor` on the California housing training set, scoring on **R²**.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 5, 7, 10, 15, 20]`.
2. For each depth, compute the 5-fold CV mean and SD using `cv_reg`.
3. Plot CV mean with SD error bars across depths.
4. Apply the same one-standard-error rule to pick the simplest competitive depth.
5. Write 3 short findings comparing this regression sweep to the classification sweep in Exercise 1 — same shape, different scale?

---

> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeRegressor with random_state=474 over depths [2,3,5,7,10,15,20] on X_train_reg, y_train_reg using cv_reg and scoring='r2'. Report mean ± SD for each depth in a DataFrame, plot mean with SD error bars, mark the highest-mean depth with a dashed line, and apply the one-standard-error rule. Convert the best CV-RMSE to USD (multiply by 100,000) and print it."
>
> **After running, verify:**
> - [ ] DataFrame has 7 rows (one per depth)
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth and one-SE-rule depth both reported
> - [ ] CV R² values are in 0.50–0.75 range
> - [ ] Best CV RMSE printed in USD


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune regression-tree depth using 5-fold CV R².
# Apply the one-standard-error rule and report best CV-RMSE in USD.


## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **Decision trees partition feature space with axis-aligned splits.** Every leaf holds a constant prediction — majority class for classification, leaf-mean for regression. Tree depth controls how finely the space is partitioned.
2. **The overfitting pattern is universal.** On both spines, training score climbs toward 1.0 while CV score plateaus. The train–CV gap is the regularization signal; pick the simplest depth whose CV mean is within one SD of the best (one-standard-error rule).
3. **Single trees can earn the win when the structure is non-linear.** On California Housing, the depth-5 regression tree dominated OLS on both CV mean and CV CI lower bound under the dominance tiebreaker — exactly the kind of interaction-heavy, non-monotonic data trees were built for. On Wisconsin breast cancer the linear reference held; the linearly-separable structure rewards the simpler model. The CV-bar comparison from Section 7 is the reusable diagnostic for that decision.
4. **Two parallel spines, two namespaces, two locked test sets.** From here through nb14, every section of every notebook will pair the classification and regression tracks under the `_clf` / `_reg` suffix convention. The discipline keeps the two cases unconfusable while you work.

**Bridge to nb12 — Random Forests:**

A single tree is high-variance: a small change in the training data can flip the root split. Random forests fix that by **averaging many trees**, each trained on a bootstrap sample with a random feature subset. The same dual-case structure continues — `RandomForestClassifier` on Wisconsin breast cancer, `RandomForestRegressor` on California housing, with paired diagnostics at every step. nb12 also introduces the **four-method feature-importance reconciliation table** that nb15 will lean on for interpretation. Bring today's depth-selection muscle memory with you — forests need to pick depth, n_estimators, and max_features under the same one-SE rule. On regression the new floor the forest has to clear is **today's tree (CV R² 0.613)**, not OLS (0.586) — a tougher bar than the linear baseline, and exactly the kind of test nb14's selection ceremony will repeat with the full five-candidate roster.

> **A question that often comes up at this point:** *"if forests beat trees on every benchmark, why teach single trees first?"* Two reasons. First, the forest is an average of trees — without understanding what one tree does, the averaging operation is opaque. Second, single trees are still the right answer when the deliverable demands a single auditable decision path (clinical screening, regulatory compliance, courtroom defense). Forests trade interpretability for stability; that trade is sometimes worth making and sometimes not, and you cannot evaluate the trade without first having seen what gets lost.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (classification depth tuning) and Exercise 2 (regression depth tuning).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 11 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a CV-CI plot with the chosen depth marked
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)
- [ ] You can defend the one-SE-rule pick on each spine in plain English

### Next Step:

- **Notebook 12** — Random Forests + Importance (Day 12)

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 8 (Tree-Based Methods), §8.1 Decision Trees. Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — Chapter on tree-based methods.
- Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Quinlan, J. R. (1986). "Induction of decision trees." *Machine Learning*, 1(1), 81–106.
- scikit-learn User Guide — [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on segmentation and decision rules.

---

<center>

**Thank you!**

</center>